<a href="https://colab.research.google.com/github/ajaramillola1985/cauca-clima-gee/blob/main/06_aptitud_riesgo_cafe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aptitud para café y riesgo de roya y muerte descendente (Phoma) en el Cauca

Este notebook usa Google Earth Engine para generar tres mapas del departamento del Cauca:

1. **Zonas aptas para café** (análisis multicriterio de clima, suelo y relieve).
2. **Riesgo de roya** (*Hemileia vastatrix*).
3. **Riesgo de muerte descendente** (*Phoma* spp.).

Los umbrales y pesos están en la celda **Criterios**. Son valores de partida basados en publicaciones de Cenicafé y deben calibrarse con datos de campo.

## 1. Conexión a Earth Engine

In [ ]:
import ee, geemap

PROYECTO_GEE = 'sig-unal-2026'

ee.Authenticate(auth_mode='notebook')   # si el token venció: ee.Authenticate(auth_mode='notebook', force=True)
ee.Initialize(project=PROYECTO_GEE)

## 2. Criterios (edita aquí umbrales y pesos)

- `trapecio: [a, b, c, d]`: puntaje 0 por debajo de `a`, sube a 1 en `b`, se mantiene hasta `c` y baja a 0 en `d`.
- `sube: [a, b]`: puntaje 0 en `a` o menos, 1 en `b` o más.
- `baja: [a, b]`: puntaje 1 en `a` o menos, 0 en `b` o más.
- Los pesos de cada mapa deben sumar 1.

In [ ]:
PERIODO = ('2015-01-01', '2025-01-01')   # periodo para lluvia, humedad y viento

APTITUD_CAFE = {
    'temperatura': {'trapecio': [16, 18, 22, 24],         'peso': 0.30},  # °C, media anual
    'lluvia':      {'trapecio': [1200, 1500, 2800, 3500], 'peso': 0.20},  # mm/año
    'altitud':     {'trapecio': [1000, 1300, 1900, 2200], 'peso': 0.15},  # m
    'pendiente':   {'baja':     [50, 100],                'peso': 0.15},  # %
    'ph':          {'trapecio': [4.5, 5.0, 5.5, 6.5],     'peso': 0.20},  # pH del suelo
}

RIESGO_ROYA = {
    'temperatura': {'trapecio': [15, 21, 25, 28], 'peso': 0.45},  # °C
    'altitud':     {'baja':     [1300, 2000],     'peso': 0.30},  # m
    'lluvia':      {'sube':     [1500, 2500],     'peso': 0.15},  # mm/año
    'humedad':     {'sube':     [78, 92],         'peso': 0.10},  # %
}

RIESGO_PHOMA = {
    'altitud':     {'sube': [1400, 1900], 'peso': 0.35},  # m
    'temperatura': {'baja': [17, 21],     'peso': 0.25},  # °C
    'amplitud':    {'sube': [9.5, 11.5],  'peso': 0.20},  # °C día-noche
    'lluvia':      {'sube': [1800, 3200], 'peso': 0.20},  # mm/año
}

# Clases de ESA WorldCover que se excluyen: construido, nieve, agua, humedal, manglar
COBERTURAS_EXCLUIDAS = [50, 70, 80, 90, 95]
EXCLUIR_AREAS_PROTEGIDAS = False

for nombre, crit in [('aptitud', APTITUD_CAFE), ('roya', RIESGO_ROYA), ('phoma', RIESGO_PHOMA)]:
    total = sum(v['peso'] for v in crit.values())
    assert abs(total - 1) < 1e-6, f'Los pesos de {nombre} suman {total}, deben sumar 1'

## 3. Área de estudio

In [ ]:
cauca = (ee.FeatureCollection('FAO/GAUL/2015/level1')
         .filter(ee.Filter.eq('ADM0_NAME', 'Colombia'))
         .filter(ee.Filter.eq('ADM1_NAME', 'Cauca')))   # nombre exacto: excluye Valle del Cauca
region = cauca.geometry()
print('Departamentos encontrados:', cauca.size().getInfo())   # debe ser 1

# División política municipal (FAO GAUL nivel 2), se usa en los mapas y en la descarga
municipios = (ee.FeatureCollection('FAO/GAUL/2015/level2')
              .filter(ee.Filter.eq('ADM0_NAME', 'Colombia'))
              .filter(ee.Filter.eq('ADM1_NAME', 'Cauca')))
print('Municipios:', municipios.size().getInfo())

## 4. Capas de clima, suelo y relieve

In [ ]:
import math

# Relieve (SRTM, 30 m)
dem = ee.Image('USGS/SRTMGL1_003')
altitud = dem.rename('altitud')
pendiente = (ee.Terrain.slope(dem).multiply(math.pi / 180).tan()
             .multiply(100).rename('pendiente'))            # en %

# Climatología WorldClim (1 km): temperatura media y amplitud térmica diaria
bio = ee.Image('WORLDCLIM/V1/BIO')
temperatura = bio.select('bio01').multiply(0.1).rename('temperatura')   # °C
amplitud = bio.select('bio02').multiply(0.1).rename('amplitud')         # °C

# Lluvia anual media (CHIRPS)
anios = ee.Number(ee.Date(PERIODO[1]).difference(ee.Date(PERIODO[0]), 'year'))
lluvia = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterDate(*PERIODO)
          .map(lambda img: img.resample('bilinear'))
          .sum().divide(anios).rename('lluvia'))

# Humedad relativa y viento (ERA5-Land mensual, ~11 km)
era = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
       .filterDate(*PERIODO).mean())
t = era.select('temperature_2m').subtract(273.15)
td = era.select('dewpoint_temperature_2m').subtract(273.15)
def presion_vapor(x):
    return x.multiply(17.625).divide(x.add(243.04)).exp()
humedad = presion_vapor(td).divide(presion_vapor(t)).multiply(100).rename('humedad')  # %
viento = (era.select('u_component_of_wind_10m').pow(2)
          .add(era.select('v_component_of_wind_10m').pow(2)).sqrt().rename('viento'))  # m/s

# pH del suelo en superficie (OpenLandMap, 250 m)
ph = (ee.Image('OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02')
      .select('b0').multiply(0.1).rename('ph'))

# Cobertura del suelo (ESA WorldCover, 10 m)
cobertura = ee.ImageCollection('ESA/WorldCover/v200').first().rename('cobertura')

capas = (ee.Image.cat([altitud, pendiente, temperatura, amplitud,
                       lluvia, humedad, viento, ph]).clip(region))
print(capas.bandNames().getInfo())

## 5. Funciones de puntaje

In [ ]:
def puntaje(img, regla):
    """Convierte una capa a un puntaje de 0 a 1 según la regla del criterio."""
    if 'trapecio' in regla:
        a, b, c, d = regla['trapecio']
        sube = img.subtract(a).divide(b - a)
        baja = ee.Image.constant(d).subtract(img).divide(d - c)
        return sube.min(baja).clamp(0, 1)
    if 'sube' in regla:
        a, b = regla['sube']
        return img.subtract(a).divide(b - a).clamp(0, 1)
    if 'baja' in regla:
        a, b = regla['baja']
        return ee.Image.constant(b).subtract(img).divide(b - a).clamp(0, 1)
    raise ValueError(f'Regla no reconocida: {regla}')

def indice(criterios, nombre):
    """Suma ponderada de los puntajes de cada variable (resultado de 0 a 1)."""
    total = ee.Image.constant(0)
    for variable, regla in criterios.items():
        total = total.add(puntaje(capas.select(variable), regla).multiply(regla['peso']))
    return total.rename(nombre).clip(region)

def clasificar(img):
    """Clases 1 a 5 con cortes en 0,2 / 0,4 / 0,6 / 0,8."""
    return (img.gte(0.2).add(img.gte(0.4)).add(img.gte(0.6)).add(img.gte(0.8))
            .add(1).rename('clase').updateMask(img.mask()))

## 6. Restricciones

In [ ]:
permitido = cobertura.remap(COBERTURAS_EXCLUIDAS, [0] * len(COBERTURAS_EXCLUIDAS), 1)

if EXCLUIR_AREAS_PROTEGIDAS:
    protegidas = (ee.FeatureCollection('WCMC/WDPA/current/polygons')
                  .filterBounds(region))
    mascara_protegidas = ee.Image.constant(1).paint(protegidas, 0)
    permitido = permitido.And(mascara_protegidas)

permitido = permitido.clip(region)

## 7. Mapa de zonas aptas para café

In [ ]:
aptitud = indice(APTITUD_CAFE, 'aptitud').updateMask(permitido)
aptitud_clase = clasificar(aptitud)

PALETA_APT = ['#d7191c', '#fdae61', '#ffffbf', '#a6d96a', '#1a9641']
ETIQ_APT = ['No apto', 'Marginal', 'Moderado', 'Apto', 'Muy apto']

from ipyleaflet import basemaps

excluido = permitido.Not().selfMask()

m = geemap.Map(basemap=basemaps.Esri.WorldTopoMap)
m.centerObject(cauca, 8)
m.addLayer(aptitud_clase, {'min': 1, 'max': 5, 'palette': PALETA_APT}, 'Aptitud café')
m.addLayer(excluido, {'palette': ['#9e9e9e']}, 'Excluido por restricción')
m.addLayer(municipios.style(color='#555555', fillColor='00000000', width=1), {}, 'Municipios')
if EXCLUIR_AREAS_PROTEGIDAS:
    m.addLayer(protegidas.style(color='#4a148c', fillColor='00000000', width=1), {}, 'Áreas protegidas')
m.addLayer(cauca.style(color='black', fillColor='00000000', width=2), {}, 'Cauca')
m.add_legend(title='Aptitud para café', legend_dict=dict(zip(ETIQ_APT, PALETA_APT)))
m

## 8. Mapas de riesgo de roya y de Phoma

El riesgo se calcula solo en la zona cafetera: aptitud ≥ 0,4 (moderado o mejor) y altitud entre 1.000 y 2.200 m. Lo demás del Cauca se muestra en gris.

In [ ]:
alt = capas.select('altitud')
zona_cafe = aptitud.gte(0.4).And(alt.gte(1000)).And(alt.lte(2200))

roya = indice(RIESGO_ROYA, 'riesgo_roya').updateMask(zona_cafe)
phoma = indice(RIESGO_PHOMA, 'riesgo_phoma').updateMask(zona_cafe)
roya_clase = clasificar(roya)
phoma_clase = clasificar(phoma)

PALETA_RIESGO = ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c']
ETIQ_RIESGO = ['Muy bajo', 'Bajo', 'Medio', 'Alto', 'Muy alto']
vis = {'min': 1, 'max': 5, 'palette': PALETA_RIESGO}

Rango de cada variable dentro de la zona cafetera (percentiles).

In [ ]:
import pandas as pd

stats = capas.updateMask(zona_cafe).reduceRegion(
    reducer=ee.Reducer.percentile([5, 25, 50, 75, 95]),
    geometry=region, scale=250, maxPixels=1e10, tileScale=4).getInfo()

filas = {}
for clave, valor in stats.items():
    variable, p = clave.rsplit('_', 1)
    filas.setdefault(variable, {})[p] = round(valor, 1)

display(pd.DataFrame(filas).T[['p5', 'p25', 'p50', 'p75', 'p95']])

Mapas de riesgo con municipios y zonas no aptas.

In [ ]:
# Todo lo que queda fuera de la zona cafetera, dentro del Cauca
no_apto = zona_cafe.unmask(0).Not().selfMask().clip(region)

COLOR_NO_APTO = '#d9d9d9'   # gris suave; prueba '#efe6d5' para un beige

def mapa_riesgo(clase, titulo):
    m = geemap.Map(basemap=basemaps.Esri.WorldTopoMap)
    m.centerObject(cauca, 8)
    m.addLayer(no_apto, {'palette': [COLOR_NO_APTO]}, 'No apto para café', True, 0.75)
    m.addLayer(clase, vis, titulo)
    m.addLayer(municipios.style(color='#555555', fillColor='00000000', width=1), {}, 'Municipios')
    m.addLayer(cauca.style(color='black', fillColor='00000000', width=2), {}, 'Cauca')
    leyenda = dict(zip(ETIQ_RIESGO, PALETA_RIESGO))
    leyenda['No apto para café'] = COLOR_NO_APTO
    m.add_legend(title=titulo, legend_dict=leyenda)
    return m

### Riesgo de roya

In [ ]:
mapa_riesgo(roya_clase, 'Riesgo de roya')

### Riesgo de Phoma

In [ ]:
mapa_riesgo(phoma_clase, 'Riesgo de Phoma')

## 9. Área por clase (hectáreas)

In [ ]:
import pandas as pd

def area_por_clase(clase, etiquetas, escala=100):
    img = ee.Image.pixelArea().divide(10000).addBands(clase)
    res = img.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='clase'),
        geometry=region, scale=escala, maxPixels=1e10, tileScale=4).getInfo()
    filas = [{'clase': etiquetas[int(g['clase']) - 1], 'hectareas': round(g['sum'])}
             for g in res['groups']]
    return pd.DataFrame(filas)

print('Aptitud para café');  display(area_por_clase(aptitud_clase, ETIQ_APT))
print('Riesgo de roya');     display(area_por_clase(roya_clase, ETIQ_RIESGO))
print('Riesgo de Phoma');    display(area_por_clase(phoma_clase, ETIQ_RIESGO))

## 10. Descargar mapas desde Colab (GeoTIFF para QGIS y PNG con límites)

Genera, dentro de la carpeta `mapas_cafe_cauca`, y descarga todo en un solo `mapas_cafe_cauca.zip`:

| Archivo | Contenido |
|---|---|
| `indices_cafe_cauca.tif` | 3 bandas con los índices de 0 a 1 (`aptitud_cafe`, `riesgo_roya`, `riesgo_phoma`); sin dato = -9999. |
| `*_clases.tif` | Un mapa por índice con las 5 clases y sus colores ya asignados (se abre con colores en QGIS). |
| `*.png` | Mapas listos para presentaciones, con límites municipales y departamental, leyenda y coordenadas. |
| `municipios_cauca.gpkg` | Límites municipales para QGIS. |

**Sobre los píxeles sin dato.** El GeoTIFF es un rectángulo que encierra al Cauca, así que todo lo que queda fuera del departamento (incluido el océano Pacífico) no tiene dato. Por eso, sobre el rectángulo completo, faltan más de la mitad de los píxeles. La celda informa el porcentaje **dentro del Cauca**, que es el que importa: para la aptitud corresponde a las coberturas excluidas y para el riesgo, a lo que queda fuera de la zona cafetera.

Primera celda: descarga desde Earth Engine (tarda 1–2 minutos).

In [ ]:
!pip install -q rasterio
import os, zipfile
import numpy as np, rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch
from rasterio.features import geometry_mask
from IPython.display import Image, display
from google.colab import files

ESCALA = 250      # metros por píxel. Para mapas finales a 100 m usa la exportación a Drive (sección 11)
NODATA = -9999    # valor que marca "sin dato" en el GeoTIFF
CARPETA = 'mapas_cafe_cauca'
BANDAS = ['aptitud_cafe', 'riesgo_roya', 'riesgo_phoma']
os.makedirs(CARPETA, exist_ok=True)

# 1) Índices (0 a 1) en un GeoTIFF de 3 bandas
archivo_indices = f'{CARPETA}/indices_cafe_cauca.tif'
if os.path.exists(archivo_indices):
    os.remove(archivo_indices)
indices = ee.Image.cat([aptitud, roya, phoma]).rename(BANDAS).unmask(NODATA).toFloat()
geemap.ee_export_image(indices, filename=archivo_indices, scale=ESCALA,
                       region=region, crs='EPSG:4326')
if not os.path.exists(archivo_indices) or os.path.getsize(archivo_indices) == 0:
    raise RuntimeError('La descarga falló. Sube ESCALA (p. ej. 500) o usa la exportación a Drive.')
with rasterio.open(archivo_indices, 'r+') as src:
    src.nodata = NODATA
    for i, banda in enumerate(BANDAS, 1):
        src.set_band_description(i, banda)

# 2) Límites del departamento y de los municipios (FAO GAUL 2015, igual que el resto del notebook)
def a_geodataframe(fc):
    gdf = geemap.ee_to_gdf(fc.map(lambda f: f.simplify(maxError=100)))
    gdf = gdf.set_crs('EPSG:4326', allow_override=True)
    gdf['geometry'] = gdf.geometry.make_valid()
    gdf = gdf.explode(index_parts=False)
    return gdf[gdf.geom_type.isin(['Polygon', 'MultiPolygon'])]

municipios_gdf = a_geodataframe(municipios)
cauca_gdf = a_geodataframe(cauca)
municipios_gdf[['ADM2_NAME', 'geometry']].to_file(f'{CARPETA}/municipios_cauca.gpkg', driver='GPKG')
print(f'✔ GeoTIFF descargado y {municipios_gdf["ADM2_NAME"].nunique()} municipios cargados')

Segunda celda: clases, mapas PNG y archivo .zip (no consume cuota de Earth Engine; puedes repetirla para cambiar colores o títulos).

In [ ]:
CORTES = [0.2, 0.4, 0.6, 0.8]   # mismos cortes que clasificar()
COLOR_GRIS = '#d9d9d9'
MAPAS = {   # banda: (título, paleta, etiquetas, qué significa el gris dentro del Cauca)
    'aptitud_cafe': ('Aptitud para café', PALETA_APT, ETIQ_APT,
                     'Excluido (agua, urbano, humedal)'),
    'riesgo_roya':  ('Riesgo de roya', PALETA_RIESGO, ETIQ_RIESGO, 'Fuera de la zona cafetera'),
    'riesgo_phoma': ('Riesgo de Phoma', PALETA_RIESGO, ETIQ_RIESGO, 'Fuera de la zona cafetera'),
}

def rgba255(color):
    return tuple(int(round(c * 255)) for c in to_rgba(color))

with rasterio.open(archivo_indices) as src:
    perfil = src.profile.copy()
    perfil.update(count=1, dtype='uint8', nodata=0, compress='lzw')
    dentro = ~geometry_mask(cauca_gdf.geometry, out_shape=(src.height, src.width),
                            transform=src.transform)
    extension = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    xmin, ymin, xmax, ymax = cauca_gdf.total_bounds
    margen = 0.05
    proporcion = 1 / np.cos(np.radians((ymin + ymax) / 2))

    for banda, nombre in enumerate(BANDAS, 1):
        titulo, paleta, etiquetas, texto_gris = MAPAS[nombre]
        v = src.read(banda)
        con_dato = (v != NODATA) & dentro
        clase = np.zeros(v.shape, dtype='uint8')                 # 0 = sin dato
        clase[con_dato] = np.digitize(v[con_dato], CORTES) + 1   # 1 a 5
        print(f'{titulo}: {100 * (~con_dato[dentro]).mean():.0f} % del Cauca en gris '
              f'({texto_gris.lower()})')

        # GeoTIFF de clases con colores, listo para QGIS
        with rasterio.open(f'{CARPETA}/{nombre}_clases.tif', 'w', **perfil) as dst:
            dst.write(clase, 1)
            dst.write_colormap(1, {0: (0, 0, 0, 0),
                                   **{i + 1: rgba255(c) for i, c in enumerate(paleta)}})

        # PNG con límites municipales y departamental
        colores = np.array([to_rgba('white', 0)] + [to_rgba(c) for c in paleta])
        imagen = colores[clase]
        imagen[dentro & ~con_dato] = to_rgba(COLOR_GRIS)

        fig, ax = plt.subplots(figsize=(9, 10))
        ax.imshow(imagen, extent=extension, interpolation='nearest')
        municipios_gdf.boundary.plot(ax=ax, color='#555555', linewidth=0.5, aspect=None)
        cauca_gdf.boundary.plot(ax=ax, color='black', linewidth=1.3, aspect=None)
        ax.set_xlim(xmin - margen, xmax + margen)
        ax.set_ylim(ymin - margen, ymax + margen)
        ax.set_aspect(proporcion)
        leyenda = [Patch(facecolor=c, edgecolor='0.4', label=e) for c, e in zip(paleta, etiquetas)]
        leyenda.append(Patch(facecolor=COLOR_GRIS, edgecolor='0.4', label=texto_gris))
        ax.legend(handles=leyenda, title=titulo, loc='lower left', fontsize=8, title_fontsize=9,
                  framealpha=0.9)
        ax.set_title(f'{titulo} – Cauca (Colombia)', fontsize=13)
        ax.set_xlabel('Longitud (°)'); ax.set_ylabel('Latitud (°)')
        png = f'{CARPETA}/{nombre}.png'
        fig.savefig(png, dpi=200, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        display(Image(filename=png, width=520))

# 3) Todo en un solo .zip
archivo_zip = 'mapas_cafe_cauca.zip'
with zipfile.ZipFile(archivo_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(CARPETA)):
        z.write(os.path.join(CARPETA, f), f)
        print('  ·', f, f'({os.path.getsize(os.path.join(CARPETA, f)) / 1024:.0f} KB)')
files.download(archivo_zip)

## 11. Exportar a Google Drive (GeoTIFF a 100 m)

Para mapas finales a mayor resolución. La tarea corre en segundo plano y deja el archivo en la carpeta `cauca_gee` de tu Drive.

In [ ]:
salida = ee.Image.cat([aptitud, roya, phoma]).toFloat()

tarea = ee.batch.Export.image.toDrive(
    image=salida, description='cauca_cafe_aptitud_riesgo',
    folder='cauca_gee', region=region, scale=100, crs='EPSG:4326', maxPixels=1e10)
tarea.start()
print('Exportación iniciada. Revisa el avance en https://code.earthengine.google.com/tasks')